In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["groq_api_key"]=os.getenv("groq_api_key")

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

In [6]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-20b")

In [7]:
agent=create_agent(
model,
checkpointer=InMemorySaver(),
middleware=[ 
SummarizationMiddleware(
model,
 trigger=("messages",10), 
keep=("messages",40)) 
]
)

In [8]:
config={"configurable":{"thread_id":"test-1"}}

In [10]:
questions=[
"what is 2+2?",
"what is 10*5?",
"what is 100/4?",
"what is 15-7?",
"what is 3*3?",
"what is 4*4?"]

for q in questions:
	response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
	print(f"Messages:{response}")
	print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='dec0c53e-c32b-47f9-8234-dbe0adea44fe'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'We need to respond to user question: "what is 2+2?" It\'s a simple math question. The answer is 4. We should respond concisely. No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 78, 'total_tokens': 137, 'completion_time': 0.073689808, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.003660915, 'prompt_tokens_details': None, 'queue_time': 0.211080362, 'total_time': 0.077350723}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d439-d779-7943-bf24-44568217171f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'outpu

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

In [18]:
@tool
def search_hotels(city:str)->str:
	""" Seach hotels - returns long response to use more tokens."""
	return f"""hotels in {city}:
1.Grand Hotel -5 star,$350/night,spa,pool,gym
2.city Inn - 4 star, $18-/night,business center
3.Budget stay-3 star,$75/night ,free wifi"""
agent=create_agent( 
model,
tools=[search_hotels],
checkpointer=InMemorySaver(),
middleware=[
SummarizationMiddleware(
model,
trigger=("tokens",550),
keep=("tokens",220))
]
)


In [15]:
config={"configurable":{"thread_id":"test-1"}}


In [ ]:
def count_tokens(messages):
	total_chars=sum(len(str(m.content)) for m in messages)
	return total_chars


In [20]:
cities=["paris","london","tokyo","newyork","dubai","singapore"]
for city in cities:
	response=agent.invoke(
{"messages":[HumanMessage(content=f"find hotels in {city}")]},
config=config )

	tokens=count_tokens(response["messages"])
	print(f"{city}:~{tokens} tokens,{len(response['messages'])} messages")
	print(f"{(response['messages'])}")

paris:~1907 tokens,5 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT  \nThe user wants a list of hotels in Singapore and is open to refining the search with additional filters.\n\n## SUMMARY  \n- The assistant invoked `search_hotels` with `{"city":"Singapore"}`.  \n- The tool returned three hotels:  \n  1. Grand Hotel – 5\u202f★, $350/night, spa, pool, gym  \n  2. City Inn – 4\u202f★, $18/night, business center  \n  3. Budget Stay – 3\u202f★, $75/night, free Wi‑Fi  \n- No filters (budget, star rating, amenities, etc.) have been applied yet.  \n- The assistant has presented the list and prompted the user to specify one or more filters (budget range, star rating, neighborhood, amenities, user rating, hotel type) for a more targeted search.\n\n## ARTIFACTS  \nNone.\n\n## NEXT STEPS  \nPrompt the user to specify filtering criteria (e.g., budget range, star rating, neighborhood, amenities, user rating, hotel type) so a refined `search_hote